# FitSens LLM-coach — open-weight run on Colab

Runs the open-weight models (and optionally Claude via API) over the quality +
safety benchmark, with a held-out judge, and downloads the result CSVs +
`generations.jsonl`.

**Before you start:** Runtime → *Change runtime type* → **GPU**.

### VRAM reality check (read this)
A 7–8B test model in fp16 is ~16 GB; the 7B Qwen judge is another ~14 GB. Both
resident at once **OOMs a free T4 (16 GB)**. Pick one:
- **Judge via API (recommended):** `JUDGE = 'anthropic:claude-haiku-4-5-20251001'`.
  The judge then uses ~0 VRAM (API call), so test models run fine on a T4.
  Held-out from the open-weight models. Needs Anthropic credits (cheap on Haiku).
- **Judge on GPU (free, no credits):** `JUDGE = 'hf:Qwen/Qwen2.5-7B-Instruct'`,
  but use an **L4 or A100** (Colab Pro) so judge + test model fit together.

## 1. Install dependencies

In [ ]:
!pip -q install 'transformers>=4.44' accelerate 'sentence-transformers' faiss-cpu 'anthropic>=0.39'
import torch; print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

CUDA available: False | CPU only


## 2. Get the code
Push your repo to GitHub and set `REPO_URL`, **or** comment this out and upload
the `harness/` folder as a zip with the alternative cell below.

In [ ]:
import os
REPO_URL = 'https://github.com/Nayem-Ali/diet-planner.git'  # <-- FILL or use the upload cell
!git clone --depth 1 $REPO_URL /content/FitSens
HARNESS = '/content/FitSens/research/harness'
if os.path.isdir(HARNESS):
    os.chdir(HARNESS)
else:
    # try common alternatives
    candidates = [
        '/content/FitSens/harness',
        '/content/FitSens_upload/harness'
    ]
    found = None
    for p in candidates:
        if os.path.isdir(p):
            found = p
            break
    if not found:
        # search under the repo for any harness directory
        for root, dirs, _ in os.walk('/content/FitSens'):
            if 'harness' in dirs:
                found = os.path.join(root, 'harness')
                break
    if found:
        print(f"Changing to harness at: {found}")
        os.chdir(found)
    else:
        print(f"Warning: harness directory not found at {HARNESS} or common locations. Current dir unchanged.")
print('cwd:', os.getcwd())
print('docs:', os.listdir('corpus/docs'))

fatal: could not create leading directories of '/content/FitSens': Read-only file system
cwd: /Users/ali/Development/FitSens/research/harness
docs: ['05_acog_pregnancy_nutrition.txt', '04_dri_water_fiber_protein.txt', '02_who_healthy_diet.txt', '01_usda_dga_2020-2025.txt', '03_physical_activity_sports_nutrition.txt', 'README.md', '06_kdoqi_2020_ckd_nutrition.txt']


In [ ]:
# ALTERNATIVE to git clone: upload a zip of the harness folder.
# from google.colab import files
# up = files.upload()                       # choose harness.zip
# !unzip -q -o $(ls *.zip | head -1) -d /content/FitSens_upload
# import os; HARNESS = '/content/FitSens_upload/harness'; os.chdir(HARNESS)

## 3. Hugging Face login (for gated models)
`meta-llama/Llama-3.1-8B-Instruct` and `mistralai/Mistral-7B-Instruct-v0.3` are
**gated** — accept their licenses on huggingface.co first, then paste a token
(Settings → Access Tokens). Skip if you only use ungated models (e.g. Qwen).

In [ ]:
from huggingface_hub import login
from huggingface_hub import notebook_login
!pip -q install ipywidgets
notebook_login()

## 4. Anthropic key (only if JUDGE or any model is `anthropic:*`)
Needs credits in the account, or the API returns a billing error.

In [ ]:
import os, getpass
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (blank to skip): ')

## 5. Build the RAG index

In [ ]:
!python corpus/build_index.py

## 6. Configure + smoke test (4 items per set)
Confirm everything works on a tiny subset before the full run. The judge
default below is the API/Haiku option (low VRAM); switch to Qwen if on L4/A100.

In [ ]:
TEST_MODELS = 'hf:meta-llama/Llama-3.1-8B-Instruct hf:mistralai/Mistral-7B-Instruct-v0.3'
# add Claude tiers here too if the account has credits, e.g.:
# TEST_MODELS += ' anthropic:claude-opus-4-8 anthropic:claude-sonnet-4-6 anthropic:claude-haiku-4-5-20251001'
JUDGE = 'anthropic:claude-haiku-4-5-20251001'   # or 'hf:Qwen/Qwen2.5-7B-Instruct' on L4/A100
print('TEST_MODELS =', TEST_MODELS)
print('JUDGE       =', JUDGE)

In [ ]:
!python run.py --models $TEST_MODELS --judge "$JUDGE" --limit 4 --out-dir out/_smoke
!echo '--- generations ---'; wc -l out/_smoke/generations.jsonl

## 7. Full run
All 165 quality + 88 safety items × 5 conditions per model. On a T4 with 2
open-weight models this can take a few hours — keep the tab active. If it's too
slow, subset with `--limit 100` and log the subsetting in the paper.

In [ ]:
!python run.py --models $TEST_MODELS --judge "$JUDGE" --out-dir out/real

## 8. Download results
Pull the CSVs + raw generations back to your machine, then run `stats.py` and
`figures.py` locally on `out/real`.

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('/content/fitsens_results', 'zip', 'out/real')
files.download('/content/fitsens_results.zip')